# 🚲 London Bike-Share Demand Analysis

> **Goal:** Explore, forecast, and spatially analyse bike-rental demand across London's bike-share network using temporal modelling, spatial statistics, and unsupervised clustering.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 2. Load Data

Two CSV files are used:
- **`london.csv`** – individual rental transactions
- **`london_stations.csv`** – station metadata (name, latitude, longitude)

In [ ]:
london   = pd.read_csv("london.csv")
stations = pd.read_csv("london_stations.csv")

In [ ]:
print(stations.head())
print(stations.columns)

In [ ]:
print(london.head())
print(london.columns)

## 3. Preprocessing

- Parse `start_rental_date_time` and `end_rental_date_time` into proper datetime objects.
- Drop rows with missing start times or station IDs.
- Floor timestamps to hourly buckets for time-series aggregation.

In [ ]:
# Parse datetime columns
london['start_time'] = pd.to_datetime(london['start_rental_date_time'], errors='coerce')
london['end_time']   = pd.to_datetime(london['end_rental_date_time'],   errors='coerce')

# Drop rows with unparseable timestamps or missing station IDs
london = london.dropna(subset=['start_time', 'start_station_id'])

# Create hourly bucket column
london['hour'] = london['start_time'].dt.floor('H')

In [ ]:
london.head()

## 4. Temporal Demand Analysis

Aggregate the transaction data into an hourly time series of total demand across all stations.

In [ ]:
# Hourly demand across all stations
london_ts = london.groupby('hour').size().rename('count')

print(f"Time series length: {london_ts.shape[0]} hourly observations")
london_ts.head()

In [ ]:
london_ts.plot(figsize=(12, 4), title='Total Hourly Demand Over Time')
plt.xlabel('Date')
plt.ylabel('Rentals')
plt.tight_layout()
plt.show()

### 4.1 STL Decomposition

STL (Seasonal-Trend decomposition using LOESS) separates the series into trend, seasonality, and residual components.

In [ ]:
from statsmodels.tsa.seasonal import STL

# Daily seasonality (period = 24 hours)
stl = STL(london_ts, period=24)
res = stl.fit()
res.plot()
plt.suptitle('STL Decomposition – Daily Seasonality', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Weekly seasonality (period = 168 hours)
stl_week = STL(london_ts, period=168)
res_week = stl_week.fit()
res_week.plot()
plt.suptitle('STL Decomposition – Weekly Seasonality', y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Average Hourly Pattern & Stationarity Test

In [ ]:
# Average demand by hour of day
hourly_pattern = london_ts.groupby(london_ts.index.hour).mean()

plt.figure(figsize=(8, 4))
plt.plot(hourly_pattern)
plt.title('London – Average Hourly Demand Pattern')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Demand')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 4))
plot_acf(london_ts, lags=200)
plt.title('ACF – London Hourly Demand')
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(london_ts)
print('ADF Statistic :', result[0])
print('p-value       :', result[1])
print('Critical Values:')
for key, value in result[4].items():
    print(f'   {key}: {value:.4f}')

## 5. Feature Engineering for Forecasting

Build a supervised learning dataset from the time series by adding:
- **Lag features:** 1 h, 24 h, 168 h
- **Calendar features:** hour of day, day of week, weekend flag
- **Rolling statistics:** 24-hour rolling mean & std
- **Peak-hour flags:** morning (7–9) and evening (16–18) peaks
- **Target:** `log1p(count)` to reduce skew

In [ ]:
ts = london_ts.to_frame()

# Lag features
ts['lag_1']   = ts['count'].shift(1)
ts['lag_24']  = ts['count'].shift(24)
ts['lag_168'] = ts['count'].shift(168)

# Calendar features
ts['hour']       = ts.index.hour
ts['dayofweek']  = ts.index.dayofweek
ts['is_weekend'] = (ts['dayofweek'] >= 5).astype(int)
ts['is_weekday'] = (ts['dayofweek'] < 5).astype(int)

# Rolling statistics
ts['rolling_mean_24'] = ts['count'].rolling(24).mean()
ts['rolling_std_24']  = ts['count'].rolling(24).std()

# Log-transform target
ts['log_count'] = np.log1p(ts['count'])

# Peak-hour flags
ts['is_morning_peak'] = ts.index.hour.isin([7, 8, 9]).astype(int)
ts['is_evening_peak'] = ts.index.hour.isin([16, 17, 18]).astype(int)

# Interaction: lag × evening peak
ts['lag24_x_peak'] = ts['lag_24'] * ts['is_evening_peak']

ts = ts.dropna()
print(ts.shape)
ts.head()

In [ ]:
# Average demand by day of week (0 = Monday)
ts['dayofweek'] = ts.index.dayofweek
ts.groupby('dayofweek')['count'].mean().plot(kind='bar', title='Avg Demand by Day of Week')
plt.xlabel('Day of Week (0=Mon)')
plt.ylabel('Avg Demand')
plt.tight_layout()
plt.show()

### 5.1 Train / Test Split

Use a chronological (shuffle=False) 75/25 split to preserve time ordering.

In [ ]:
from sklearn.model_selection import train_test_split

X = ts.drop(columns=['count', 'log_count'])
y = ts['log_count']   # log-scale target

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## 6. Predictive Modelling

Four models are trained and evaluated using RMSE on the original (exp-transformed) scale:
1. Linear Regression
2. Ridge Regression
3. Random Forest
4. XGBoost

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lin = LinearRegression()
lin.fit(X_train, y_train)

pred_lin_log = lin.predict(X_test)
y_pred_lin   = np.expm1(pred_lin_log)
y_true       = np.expm1(y_test)

rmse_lin = np.sqrt(mean_squared_error(y_true, y_pred_lin))
print(f'Linear Regression RMSE: {rmse_lin:.2f}')

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = np.expm1(ridge.predict(X_test))
rmse_ridge   = np.sqrt(mean_squared_error(y_true, y_pred_ridge))
print(f'Ridge Regression RMSE: {rmse_ridge:.2f}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = np.expm1(rf.predict(X_test))
rmse_rf   = np.sqrt(mean_squared_error(y_true, y_pred_rf))
print(f'Random Forest RMSE: {rmse_rf:.2f}')

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1)
xgb.fit(X_train, y_train)

y_pred_xgb = np.expm1(xgb.predict(X_test))
rmse_xgb   = np.sqrt(mean_squared_error(y_true, y_pred_xgb))
print(f'XGBoost RMSE: {rmse_xgb:.2f}')

### 6.1 Model Diagnostics – Random Forest

In [ ]:
# Actual vs Predicted (first 500 test points)
plt.figure(figsize=(12, 4))
plt.plot(y_true.index[:500],    y_true[:500],      label='Actual')
plt.plot(y_true.index[:500],    y_pred_rf[:500],   label='Predicted')
plt.title('Random Forest – Actual vs Predicted (First 500 Hours)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: actual vs predicted
plt.figure(figsize=(5, 5))
plt.scatter(y_true, y_pred_rf, alpha=0.3)
plt.plot([y_true.min(), y_true.max()],
         [y_true.min(), y_true.max()], color='red', label='Perfect')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted – Scatter')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals over time
residuals = y_true - y_pred_rf

plt.figure(figsize=(10, 4))
plt.plot(y_true.index, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuals Over Time')
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
plt.figure(figsize=(6, 4))
plt.hist(residuals, bins=50)
plt.title('Residual Distribution')
plt.xlabel('Error')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 4))
plot_acf(residuals, lags=50)
plt.title('ACF of Residuals')
plt.tight_layout()
plt.show()

### 6.2 Feature Importance

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
feat_imp.plot(kind='bar')
plt.title('Feature Importance – Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feat_imp)

## 7. Spatial Analysis

Aggregate demand by station and merge with station coordinates to enable spatial modelling.

In [ ]:
# Total trips per station
station_avg = (
    london.groupby('start_station_id')
    .size()
    .reset_index(name='total_trips')
)

# Average hourly demand per station
station_avg['avg_demand'] = station_avg['total_trips'] / london_ts.shape[0]

# Merge with station coordinates
london_spatial = pd.merge(
    station_avg, stations,
    left_on='start_station_id', right_on='station_id'
)
london_spatial = london_spatial.dropna(subset=['longitude', 'latitude'])
print(london_spatial.shape)
london_spatial.head()

In [ ]:
# Spatial demand heatmap
plt.figure(figsize=(8, 6))
sc = plt.scatter(
    london_spatial['longitude'], london_spatial['latitude'],
    c=london_spatial['avg_demand'], cmap='viridis', s=30
)
plt.colorbar(sc, label='Avg Hourly Demand')
plt.title('Spatial Distribution of Station Demand')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

### 7.1 Spatial Autocorrelation – Moran's I

Test whether high-demand stations cluster spatially (positive autocorrelation).

In [ ]:
import libpysal
from esda.moran import Moran

coords = london_spatial[['longitude', 'latitude']].values
w = libpysal.weights.KNN.from_array(coords, k=5)
w.transform = 'r'

moran = Moran(london_spatial['avg_demand'], w)
print(f"Moran's I : {moran.I:.4f}")
print(f"p-value   : {moran.p_sim:.4f}")

In [ ]:
# Moran scatter plot
y_sp  = london_spatial['avg_demand'].values
lag_y = libpysal.weights.spatial_lag.lag_spatial(w, y_sp)

b, a = np.polyfit(y_sp, lag_y, 1)

plt.figure(figsize=(6, 6))
plt.scatter(y_sp, lag_y, alpha=0.5)
plt.plot(y_sp, a + b * y_sp, color='red')
plt.xlabel('Demand')
plt.ylabel('Spatial Lag of Demand')
plt.title("Moran Scatter Plot")
plt.tight_layout()
plt.show()

### 7.2 LISA – Local Indicators of Spatial Association

Identify statistically significant local clusters (High-High, Low-Low) and outliers (High-Low, Low-High).

In [ ]:
from esda.moran import Moran_Local

# Standardise demand
y_std = (london_spatial['avg_demand'] - london_spatial['avg_demand'].mean()) / london_spatial['avg_demand'].std()

lisa = Moran_Local(y_std, w)

london_spatial['lisa_cluster'] = lisa.q
london_spatial['p_lisa']       = lisa.p_sim

cluster_labels = {1: 'High-High', 2: 'Low-Low', 3: 'High-Low', 4: 'Low-High'}
london_spatial['lisa_label'] = london_spatial['lisa_cluster'].map(cluster_labels)

sig = london_spatial[london_spatial['p_lisa'] < 0.05]
print(sig['lisa_label'].value_counts())

In [ ]:
colors = {1: 'red', 2: 'blue', 3: 'orange', 4: 'purple'}
labels = cluster_labels

plt.figure(figsize=(8, 6))
plt.scatter(
    london_spatial['longitude'], london_spatial['latitude'],
    color='lightgrey', label='Not significant', s=20
)

for cluster in [1, 2, 3, 4]:
    subset = sig[sig['lisa_cluster'] == cluster]
    plt.scatter(
        subset['longitude'], subset['latitude'],
        c=colors[cluster], label=labels[cluster], s=30
    )

plt.legend()
plt.title('LISA Clusters (p < 0.05)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

### 7.3 Spatial Feature Engineering

Add distance-from-centre and neighbourhood demand features to improve spatial models.

In [ ]:
from sklearn.neighbors import NearestNeighbors

center_lon = london_spatial['longitude'].mean()
center_lat = london_spatial['latitude'].mean()

# Euclidean distance from city centroid
london_spatial['dist_center'] = np.sqrt(
    (london_spatial['longitude'] - center_lon)**2 +
    (london_spatial['latitude']  - center_lat)**2
)

# Average demand of 5 nearest neighbours
coords = london_spatial[['longitude', 'latitude']].values
nbrs = NearestNeighbors(n_neighbors=5).fit(coords)
_, indices = nbrs.kneighbors(coords)

london_spatial['neighbor_demand'] = [
    london_spatial.iloc[neigh]['avg_demand'].mean()
    for neigh in indices
]

# Colocation: fraction of neighbours with above-75th-percentile demand
threshold = london_spatial['avg_demand'].quantile(0.75)
london_spatial['is_high'] = (london_spatial['avg_demand'] >= threshold).astype(int)

london_spatial['neighbor_high_ratio'] = [
    london_spatial.iloc[neigh]['is_high'].mean()
    for neigh in indices
]

print('Colocation correlation:')
print(london_spatial[['avg_demand', 'neighbor_high_ratio']].corr())

london_spatial.head()

### 7.4 Spatial Demand Prediction

Train models to predict per-station average demand from purely spatial/neighbourhood features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

X_sp = london_spatial[['longitude', 'latitude', 'dist_center', 'neighbor_demand']]
y_sp = london_spatial['avg_demand']

X_tr, X_te, y_tr, y_te = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42)

results = {}
for name, model in [
    ('Linear',        LinearRegression()),
    ('Ridge',         Ridge(alpha=1.0)),
    ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('XGBoost',       XGBRegressor(n_estimators=100, learning_rate=0.1)),
]:
    model.fit(X_tr, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    results[name] = rmse
    print(f'{name:15s} RMSE: {rmse:.4f}')

# Keep RF for feature importance
rf_sp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_sp.fit(X_tr, y_tr)

In [ ]:
feat_imp_sp = pd.Series(rf_sp.feature_importances_, index=X_sp.columns).sort_values(ascending=False)

plt.figure(figsize=(6, 4))
feat_imp_sp.plot(kind='bar')
plt.title('Feature Importance – Spatial Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Spatial Clustering

Three K-Means clustering strategies are compared:
1. **Geographic** – longitude + latitude only
2. **Demand** – average demand + neighbourhood demand
3. **Combined** – scaled geo + demand features

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

k = 4

X_geo    = london_spatial[['longitude', 'latitude']]
X_demand = london_spatial[['avg_demand', 'neighbor_demand']]
X_comb   = london_spatial[['longitude', 'latitude', 'avg_demand', 'neighbor_demand']]

scaler   = StandardScaler()
X_comb_n = scaler.fit_transform(X_comb)

london_spatial['cluster_geo']    = KMeans(n_clusters=k, random_state=42).fit_predict(X_geo)
london_spatial['cluster_demand'] = KMeans(n_clusters=k, random_state=42).fit_predict(X_demand)
london_spatial['cluster_scaled'] = KMeans(n_clusters=k, random_state=42).fit_predict(X_comb_n)

print('--- Avg demand per cluster ---')
for col in ['cluster_geo', 'cluster_demand', 'cluster_scaled']:
    print(f'\n{col}:')
    print(london_spatial.groupby(col)['avg_demand'].mean().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('cluster_geo',    'tab10',  'Geographic Clusters'),
    ('cluster_demand', 'viridis','Demand Clusters'),
    ('cluster_scaled', 'Set1',   'Spatial + Demand Clusters'),
]

for ax, (col, cmap, title) in zip(axes, configs):
    ax.scatter(
        london_spatial['longitude'], london_spatial['latitude'],
        c=london_spatial[col], cmap=cmap, s=20
    )
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('K-Means Clustering Comparison (k=4)', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Behavioural Clustering – Station Temporal Profiles

Cluster stations by their **average hourly demand profile** (0–23 h) to identify distinct usage archetypes (e.g., commuter hub vs. leisure station).

In [ ]:
# Build station × hour profile matrix
london['start_rental_date_time'] = pd.to_datetime(london['start_rental_date_time'], errors='coerce')
london = london.dropna(subset=['start_rental_date_time'])
london['hour'] = london['start_rental_date_time'].dt.floor('H')

london_demand = (
    london.groupby(['start_station_id', 'hour'])
    .size()
    .reset_index(name='count')
    .rename(columns={'start_station_id': 'station_id'})
)

# Complete grid (station × hour)
all_hours    = pd.date_range(london_demand['hour'].min(), london_demand['hour'].max(), freq='H')
all_stations = london_demand['station_id'].unique()
full_index   = pd.MultiIndex.from_product([all_stations, all_hours], names=['station_id', 'hour'])

london_complete = (
    pd.DataFrame(index=full_index).reset_index()
    .merge(london_demand, on=['station_id', 'hour'], how='left')
)
london_complete['count'] = london_complete['count'].fillna(0)
london_complete = london_complete.merge(stations, on='station_id', how='left')

print(london_complete.shape)
london_complete.head()

In [ ]:
# Pivot to (station, hour-of-day) mean demand matrix
station_hour = london_complete.copy()
station_hour['hour_only'] = station_hour['hour'].dt.hour

profile = (
    station_hour.groupby(['station_id', 'hour_only'])['count']
    .mean()
    .unstack()
    .fillna(0)
)

print(f'Profile matrix shape: {profile.shape}  (stations × 24 hours)')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

profile_scaled = StandardScaler().fit_transform(profile)

kmeans = KMeans(n_clusters=4, random_state=42)
profile['cluster_behavior'] = kmeans.fit_predict(profile_scaled)

In [ ]:
plt.figure(figsize=(10, 5))
for c in range(4):
    mean_pat = profile[profile['cluster_behavior'] == c].drop(columns=['cluster_behavior']).mean()
    plt.plot(mean_pat, label=f'Cluster {c}')

plt.title('Behavioural Clusters – Mean Hourly Demand Profile')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Demand')
plt.legend()
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
# Map behavioural clusters geographically
profile_geo = profile.reset_index().merge(stations, on='station_id')

plt.figure(figsize=(8, 6))
plt.scatter(
    profile_geo['longitude'], profile_geo['latitude'],
    c=profile_geo['cluster_behavior'], cmap='tab10', s=30
)
plt.title('Behavioural Clusters – Geographic Distribution')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation: spatial cluster vs behavioural cluster
london_behavior = profile_geo.rename(columns={'cluster_behavior': 'cluster_behavior'})
print(pd.crosstab(london_spatial['cluster_geo'], london_behavior['cluster_behavior']))

In [ ]:
# Average demand by behavioural cluster
combined = pd.merge(
    london_behavior,
    london_spatial[['station_id', 'avg_demand']],
    on='station_id'
)
print('Avg demand per behavioural cluster:')
print(combined.groupby('cluster_behavior')['avg_demand'].mean().round(4))

## 10. Origin–Destination (OD) Matrix

Build an OD matrix to understand which station-to-station flows dominate the network.

In [ ]:
od = london.groupby(['start_station_id', 'end_station_id']).size().reset_index(name='count')

od_matrix = od.pivot(
    index='start_station_id',
    columns='end_station_id',
    values='count'
).fillna(0)

print(f'OD Matrix shape: {od_matrix.shape}')

In [ ]:
# Top 10 station-to-station flows
top_flows = od.sort_values('count', ascending=False).head(10)
print(top_flows)

In [ ]:
# Heatmap of a 50×50 station sample
sample = od_matrix.iloc[:50, :50]

plt.figure(figsize=(8, 6))
sns.heatmap(sample, cmap='viridis', xticklabels=False, yticklabels=False)
plt.title('OD Matrix (50×50 Station Sample)')
plt.tight_layout()
plt.show()

## 11. Symbolic Sequence Mining

Encode the hourly time series as a symbolic sequence (L / M / H) and mine frequent daily sub-patterns.

In [ ]:
# Normalise series
ts_norm  = (london_ts - london_ts.mean()) / london_ts.std()
ts_df    = ts_norm.to_frame(name='norm')

# Encode as Low / Medium / High symbols
def to_symbol(x):
    if x < -0.5: return 'L'
    elif x < 0.5: return 'M'
    else:         return 'H'

ts_df['symbol'] = ts_df['norm'].apply(to_symbol)
ts_df['date']   = ts_df.index.date
ts_df['hour']   = ts_df.index.hour

ts_df.head()

In [ ]:
# Build daily symbol sequences
daily_seq = ts_df.groupby('date')['symbol'].apply(list)

print('Example sequence (first day):')
print(daily_seq.iloc[0])

In [ ]:
from collections import Counter

def get_subseq(seq, k=3):
    return [''.join(seq[i:i+k]) for i in range(len(seq) - k + 1)]

# Length-3 patterns
patterns_3 = Counter()
for seq in daily_seq:
    patterns_3.update(get_subseq(seq, k=3))

print('Top 10 length-3 patterns:')
for p, cnt in patterns_3.most_common(10):
    print(f'  {p}  →  {cnt:,}')

In [ ]:
# Length-4 patterns
patterns_4 = Counter()
for seq in daily_seq:
    patterns_4.update(get_subseq(seq, k=4))

print('Top 10 length-4 patterns:')
for p, cnt in patterns_4.most_common(10):
    print(f'  {p}  →  {cnt:,}')